## Налаштування та автоматичний розрахунок центру

In [ ]:
import numpy as np
import os

# 1. НАЛАШТУВАННЯ ПАПОК
INPUT_DIR = "input"
OUTPUT_DIR = "output"

# Автоматично створюємо папку output, якщо її ще не існує
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. ВХІДНІ ДАНІ (мають лежати в папці input)
RECEPTOR_FILE = os.path.join(INPUT_DIR, "receptor.pdb")
LIGAND_FILE = os.path.join(INPUT_DIR, "ligand.pdb")

# Номери амінокислот кишені
ACTIVE_RESIDUES = [83, 85, 87, 88, 95, 99, 102, 103, 105, 106, 107, 108, 109, 
                   132, 133, 168, 173, 174, 177, 178, 180, 181, 182, 184, 185, 
                   198, 201, 202, 204, 205, 208]

# Функція розрахунку центру
def calculate_pocket_center(pdb_file, active_res_list):
    coords = []
    if not os.path.exists(pdb_file):
        raise FileNotFoundError(f"Файл {pdb_file} не знайдено! Перевірте папку {INPUT_DIR}.")
        
    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith("ATOM"):
                res_num = int(line[22:26].strip())
                if res_num in active_res_list:
                    x = float(line[30:38].strip())
                    y = float(line[38:46].strip())
                    z = float(line[46:54].strip())
                    coords.append([x, y, z])
                    
    center = np.mean(coords, axis=0)
    return center[0], center[1], center[2]

print("🔍 Аналізуємо рецептор та шукаємо центр кишені...")
CENTER_X, CENTER_Y, CENTER_Z = calculate_pocket_center(RECEPTOR_FILE, ACTIVE_RESIDUES)

print(f"✅ Центр кишені знайдено: X={CENTER_X:.3f}, Y={CENTER_Y:.3f}, Z={CENTER_Z:.3f}")

# 3. ШЛЯХИ ДЛЯ КОНВЕРТОВАНИХ ФАЙЛІВ (підуть в папку output)
RECEPTOR_PDBQT = os.path.join(OUTPUT_DIR, "receptor.pdbqt")
LIGAND_PDBQT = os.path.join(OUTPUT_DIR, "ligand.pdbqt")

## Конвертація у PDBQT

In [ ]:
print(f"🔄 Конвертую рецептор у PDBQT...")
!obabel "{RECEPTOR_FILE}" -O "{RECEPTOR_PDBQT}" -p 7.4 -xr

print(f"🔄 Конвертую ліганд у PDBQT...")
!obabel "{LIGAND_FILE}" -O "{LIGAND_PDBQT}" -p 7.4

print(f"✅ Конвертація завершена! Файли збережено у папку '{OUTPUT_DIR}'")

## Запуск серії докінгів (щоб обрати найкращий)

In [ ]:
import subprocess
import numpy as np

BOX_SIZES = [16, 20, 24]
EXHAUSTIVENESS = 8
REPLICATES = 3

results_summary = {}
absolute_best_affinity = 0
absolute_best_file = ""

# Створюємо загальний лог-файл в папці output
MASTER_LOG_PATH = os.path.join(OUTPUT_DIR, "ALL_DOCKING_LOGS.txt")
with open(MASTER_LOG_PATH, "w") as master_log:
    master_log.write("=== ЗВЕДЕНИЙ ЗВІТ ДОКІНГУ ===\n\n")

print(f"🧬 Починаємо масштабний скринінг...")

for size in BOX_SIZES:
    print(f"\n🚀 Тестуємо розмір Box = {size}x{size}x{size}...")
    results_summary[size] = []
    
    config_name = os.path.join(OUTPUT_DIR, f"config_box_{size}.txt")
    config_content = f"""receptor = {RECEPTOR_PDBQT}
ligand = {LIGAND_PDBQT}
center_x = {CENTER_X:.3f}
center_y = {CENTER_Y:.3f}
center_z = {CENTER_Z:.3f}
size_x = {size}
size_y = {size}
size_z = {size}
exhaustiveness = {EXHAUSTIVENESS}
"""
    with open(config_name, "w") as f:
        f.write(config_content)
        
    for rep in range(1, REPLICATES + 1):
        out_name = os.path.join(OUTPUT_DIR, f"results_box_{size}_rep_{rep}.pdbqt")
        log_name = os.path.join(OUTPUT_DIR, f"log_box_{size}_rep_{rep}.txt")
        
        # Запуск Vina
        subprocess.run(f"vina --config \"{config_name}\" --out \"{out_name}\" > \"{log_name}\"", shell=True)
        
        # Читаємо індивідуальний лог
        with open(log_name, 'r') as f:
            log_content = f.read()
            
        # Записуємо його у наш зведений текстовий документ
        with open(MASTER_LOG_PATH, "a") as master_log:
            master_log.write(f"\n--- Box: {size}x{size}x{size}, Повтор: {rep} ---\n")
            master_log.write(log_content)
            
        # Шукаємо енергію
        for line in log_content.split('\n'):
            if "   1 " in line and line.strip().startswith("1"):
                best_affinity = float(line.split()[1])
                results_summary[size].append(best_affinity)
                print(f"   Повтор {rep}: {best_affinity} kcal/mol")
                
                if best_affinity < absolute_best_affinity:
                    absolute_best_affinity = best_affinity
                    absolute_best_file = out_name
                break
                
    # Статистика розміру
    mean_aff = np.mean(results_summary[size])
    std_aff = np.std(results_summary[size])
    stat_text = f"📊 Статистика Box {size}: Середнє = {mean_aff:.2f} ± {std_aff:.2f} kcal/mol"
    print(stat_text)
    
    with open(MASTER_LOG_PATH, "a") as master_log:
        master_log.write(f"\n{stat_text}\n\n")

# Фінальні підсумки
best_size_by_mean = min(results_summary.keys(), key=lambda k: np.mean(results_summary[k]))
best_mean = np.mean(results_summary[best_size_by_mean])

summary_text = f"""
==================================================
🏆 НАЙКРАЩИЙ РОЗМІР: Box {best_size_by_mean} (Середня енергія: {best_mean:.2f} kcal/mol)
🥇 АБСОЛЮТНО КРАЩА ПОЗА лежить у файлі: {absolute_best_file} ({absolute_best_affinity} kcal/mol)
==================================================
"""
print(summary_text)

# Записуємо підсумок у кінець зведеного логу
with open(MASTER_LOG_PATH, "a") as master_log:
    master_log.write(summary_text)
print(f"📁 Всі детальні логи збережено у: {MASTER_LOG_PATH}")

## Витягування фінального (найкращого) комплексу

In [ ]:
COMPLEX_PDB = os.path.join(OUTPUT_DIR, "final_best_complex.pdb")
BEST_POSE_PDB = os.path.join(OUTPUT_DIR, "best_pose.pdb")

print(f"Створюємо фінальний комплекс з абсолютно найкращого результату...")

# Витягуємо позу переможця
!obabel -ipdbqt "{absolute_best_file}" -f 1 -l 1 -opdb -O "{BEST_POSE_PDB}"

# Склеюємо рецептор і ліганд
with open(COMPLEX_PDB, 'w') as complex_file:
    with open(RECEPTOR_FILE, 'r') as r_file:
        for line in r_file:
            if not line.startswith("END"):
                complex_file.write(line)
                
    with open(BEST_POSE_PDB, 'r') as l_file:
        for line in l_file:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                complex_file.write(line)
                
    complex_file.write("END\n")

print(f"✅ Готово! Збережено фінальний комплекс: {COMPLEX_PDB}")

## Data Visualization

In [ ]:
import matplotlib.pyplot as plt

# Беремо дані з нашого попереднього розрахунку
sizes = list(results_summary.keys())
means = [np.mean(results_summary[s]) for s in sizes]
stds = [np.std(results_summary[s]) for s in sizes]

plt.figure(figsize=(8, 5))
# Будуємо стовпчики з похибками (yerr)
plt.bar([str(s) for s in sizes], means, yerr=stds, capsize=10, 
        color='#4C72B0', edgecolor='black', alpha=0.8)

plt.xlabel('Розмір Grid Box (Å)', fontsize=12)
plt.ylabel('Енергія зв\'язування (kcal/mol)', fontsize=12)
plt.title('Оптимізація розміру сітки для докінгу BI-224436', fontsize=14)

# Перевертаємо вісь Y, бо в докінгу чим "нижче" (більше з мінусом), тим краще
plt.gca().invert_yaxis() 
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Зберігаємо графік у високій якості
plot_path = os.path.join(OUTPUT_DIR, "optimization_plot.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"✅ Графік збережено у: {plot_path}")

plt.show()

## Інтерактивне 3D

In [ ]:
# Встановлюємо бібліотеку (якщо її ще немає)
!pip install py3Dmol -q
import py3Dmol

print("🎨 Завантажуємо інтерактивну 3D модель найкращого комплексу...")

# Створюємо вікно перегляду
view = py3Dmol.view(width=800, height=500)

# Завантажуємо наш найкращий комплекс
with open(COMPLEX_PDB, 'r') as f:
    view.addModel(f.read(), 'pdb')

# Налаштовуємо стиль: Рецептор - сіра стрічка, Ліганд - яскраві палички
view.setStyle({'chain': 'A'}, {'cartoon': {'color': 'lightgray'}})
# Оскільки ліганд зазвичай не має назви ланцюга або називається UNL/LIG
view.setStyle({'resn': 'L3D'}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.2}}) 
view.setStyle({'hetflag': True}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.2}})

# Наближаємо камеру до ліганда
view.zoomTo({'hetflag': True})
view.show()